# Глубокое обучение с подкреплением

**AI4 · Other Topics** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~15 min

## Введение

Обучение с подкреплением (RL) считается одним из основных парадигм машинного обучения наряду с обучением с учителем и без учителя. Если в обучении с учителем мы опираемся на набор данных с известными результатами, то RL основано на **обучении через действие**. Например, когда мы впервые видим компьютерную игру, мы начинаем играть, даже не зная правил, и вскоре можем улучшить свои навыки просто благодаря процессу игры и корректировке своего поведения.


Для выполнения RL нам нужно:

* **Среда** или **симулятор**, который задает правила игры. Мы должны иметь возможность проводить эксперименты в симуляторе и наблюдать результаты.
* Некоторая **функция вознаграждения**, которая указывает, насколько успешным был наш эксперимент. В случае обучения игре в компьютерную игру вознаграждением будет наш итоговый счет.

На основе функции вознаграждения мы должны быть способны корректировать свое поведение и улучшать свои навыки, чтобы в следующий раз играть лучше. Основное отличие RL от других типов машинного обучения заключается в том, что в RL мы обычно не знаем, выиграли мы или проиграли, до тех пор, пока не закончим игру. Таким образом, мы не можем сказать, является ли определенный ход хорошим или плохим — вознаграждение мы получаем только в конце игры.

Во время RL мы обычно проводим множество экспериментов. В каждом эксперименте нам нужно находить баланс между следованием оптимальной стратегии, которую мы изучили до сих пор (**эксплуатация**), и исследованием новых возможных состояний (**исследование**).



## OpenAI Gym

Отличным инструментом для RL является [OpenAI Gym](https://gymnasium.farama.org/) — **среда симуляции**, которая может моделировать множество различных сред, начиная от игр Atari и заканчивая физикой балансировки шеста. Это одна из самых популярных сред симуляции для обучения алгоритмов обучения с подкреплением, поддерживаемая [OpenAI](https://openai.com/).

> **Note**: Вы можете увидеть все доступные среды OpenAI Gym [здесь](https://gymnasium.farama.org/environments/classic_control/).



## Балансировка CartPole

Вы, вероятно, видели современные устройства для балансировки, такие как *Segway* или *гироскутеры*. Они способны автоматически балансировать, регулируя свои колеса в ответ на сигнал от акселерометра или гироскопа. В этом разделе мы научимся решать похожую задачу — балансировку шеста. Это похоже на ситуацию, когда цирковой артист балансирует шест на своей руке, но в данном случае балансировка происходит только в одной плоскости.

Упрощенная версия задачи балансировки известна как проблема **CartPole**. В мире CartPole у нас есть горизонтальная платформа, которая может двигаться влево или вправо, и цель — сбалансировать вертикальный шест на вершине платформы, пока она движется.

![a cartpole](ms-assets/ai/cartpole.f52a67f27e058170.webp)

Чтобы создать и использовать эту среду, нам нужно всего несколько строк кода на Python:



In [ ]:
import gym
env = gym.make("CartPole-v1")

env.reset()
done = False
total_reward = 0
while not done:
   env.render()
   action = env.action_space.sample()
   observaton, reward, done, info = env.step(action)
   total_reward += reward

print(f"Total reward: {total_reward}")

Каждая среда может быть доступна точно таким же образом:
* `env.reset` начинает новый эксперимент
* `env.step` выполняет шаг симуляции. Он принимает **действие** из **пространства действий** и возвращает **наблюдение** (из пространства наблюдений), а также вознаграждение и флаг завершения.

В приведенном выше примере на каждом шаге выполняется случайное действие, поэтому продолжительность эксперимента очень короткая:

![non-balancing cartpole](ms-assets/ai/lessons/6-Other/22-DeepRL/images/cartpole-nobalance.gif)

Цель алгоритма RL — обучить модель, так называемую **политику** π, которая будет возвращать действие в ответ на данное состояние. Мы также можем рассматривать политику как вероятностную, например, для любого состояния *s* и действия *a* она будет возвращать вероятность π(*a*|*s*), что мы должны выполнить *a* в состоянии *s*.



## Алгоритм Policy Gradients

Самый очевидный способ моделирования политики — создание нейронной сети, которая будет принимать состояния в качестве входных данных и возвращать соответствующие действия (или скорее вероятности всех действий). В некотором смысле это похоже на обычную задачу классификации, с одним важным отличием — мы заранее не знаем, какие действия следует предпринимать на каждом из шагов.

Идея заключается в оценке этих вероятностей. Мы строим вектор **накопленных вознаграждений**, который показывает наш общий результат на каждом шаге эксперимента. Мы также применяем **дисконтирование вознаграждений**, умножая вознаграждение, полученное через t шагов, на γ^t (γ=0.99): чем позже награда, тем меньше её вес. Затем мы усиливаем те шаги на пути эксперимента, которые приносят больше вознаграждений.

> Узнайте больше об алгоритме Policy Gradient и посмотрите его в действии в [примере ноутбука](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/6-Other/22-DeepRL/CartPole-RL-TF.ipynb).



## Алгоритм Actor-Critic

Улучшенная версия подхода Policy Gradients называется **Actor-Critic**. Основная идея заключается в том, что нейронная сеть будет обучена возвращать две вещи:

* Политику, которая определяет, какое действие предпринять. Эта часть называется **actor**.
* Оценку общего вознаграждения, которое мы можем ожидать в данном состоянии — эта часть называется **critic**.

В некотором смысле эта архитектура напоминает [GAN](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/10-GANs/README.md), где у нас тоже две сети, но они не соревнуются, а дополняют друг друга. В модели Actor-Critic актор предлагает действие, которое нужно предпринять, а критик пытается быть критичным и оценить результат. Однако наша цель — обучить эти сети в унисон.

Поскольку мы знаем как реальные накопленные вознаграждения, так и результаты, возвращаемые критиком во время эксперимента, относительно легко построить функцию потерь, которая минимизирует разницу между ними. Это даст нам **потери критика**. Мы можем вычислить **потери актора**, используя тот же подход, что и в алгоритме Policy Gradient.

После выполнения одного из этих алгоритмов мы можем ожидать, что наш CartPole будет вести себя следующим образом:

![a balancing cartpole](ms-assets/ai/lessons/6-Other/22-DeepRL/images/cartpole-balance.gif)



## Упражнения: Policy Gradients и Actor-Critic RL

Продолжите обучение в следующих ноутбуках:

* [RL в TensorFlow](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/6-Other/22-DeepRL/CartPole-RL-TF.ipynb)
* [RL в PyTorch](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/6-Other/22-DeepRL/CartPole-RL-PyTorch.ipynb)



## Другие задачи RL

Обучение с подкреплением сегодня является быстро развивающейся областью исследований. Некоторые интересные примеры обучения с подкреплением:

* Обучение компьютера играть в **игры Atari**. Сложность этой задачи заключается в том, что у нас нет простого состояния, представленного в виде вектора, а есть скриншот — и нам нужно использовать CNN, чтобы преобразовать изображение экрана в вектор признаков или извлечь информацию о вознаграждении. Игры Atari доступны в Gym.
* Обучение компьютера играть в настольные игры, такие как шахматы и го. Недавно программы, такие как **Alpha Zero**, были обучены с нуля двумя агентами, играющими друг против друга и улучшающимися на каждом шаге.
* В промышленности RL используется для создания систем управления на основе симуляции. Сервис [Bonsai](https://azure.microsoft.com/services/project-bonsai/?WT.mc_id=academic-77998-cacaste) специально разработан для этого.



## Заключение

Мы научились обучать агентов достигать хороших результатов, просто предоставляя им функцию вознаграждения, которая определяет желаемое состояние игры, и давая им возможность разумно исследовать пространство поиска. Мы успешно попробовали два алгоритма и достигли хорошего результата за относительно короткий период времени. Однако это лишь начало вашего пути в RL, и вам определенно стоит рассмотреть возможность прохождения отдельного курса, если вы хотите углубиться в эту тему.



## Задание

Изучите приложения, перечисленные в разделе «Другие задачи RL», и попробуйте реализовать одно из них!



## Обзор и самостоятельное изучение

Узнайте больше о классическом обучении с подкреплением в нашем [курсе «Машинное обучение для начинающих»](https://github.com/microsoft/ML-For-Beginners/blob/main/8-Reinforcement/README.md).

Посмотрите [это отличное видео](https://www.youtube.com/watch?v=qv6UVOQ0F44) о том, как компьютер может научиться играть в Super Mario.



## Задание: Обучите Mountain Car

Ваша цель в этом задании — обучить другую среду Gym — [Mountain Car](https://gymnasium.farama.org/).



---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — Softmax policy with temperature

●●○ · ~10 min

> A softmax (Boltzmann) policy turns preference scores into action probabilities: π(a|s) = exp(h_a/τ) / Σ_b exp(h_b/τ). Temperature τ controls exploration: τ→0 approaches greedy, τ→∞ approaches uniform. Always subtract the max before exp() to avoid overflow.

Implement `softmax_policy(P, tau)` for a 2-D preference matrix `P` of shape (n_states, n_actions): return the row-wise softmax of `P / tau`, i.e. each row sums to 1. Use the subtract-the-max stabilization.

<details><summary>Hint</summary>

z = P/tau; z = z - z.max(axis=-1, keepdims=True); e = np.exp(z); return e / e.sum(axis=-1, keepdims=True).

</details>


In [ ]:
import numpy as np

def softmax_policy(P, tau):
    P = np.asarray(P, dtype=float)
    # your code
    return np.zeros_like(P)

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n = int(rng.integers(2, 6)); k = int(rng.integers(2, 7))\n    P = rng.normal(0, 2, (n, k))\n    if rng.random() < 0.4:\n        P = P + float(rng.uniform(900, 1400))   # stabilization is required: naive exp overflows\n    return {\"P\": P, \"tau\": float(rng.uniform(0.3, 2.5))}\n\ndef softmax_policy(P, tau):\n    z = np.asarray(P, dtype=float) / tau\n    z = z - z.max(axis=-1, keepdims=True)\n    e = np.exp(z)\n    return e / e.sum(axis=-1, keepdims=True)", "softmax_policy", 20)

### Task 2 — REINFORCE log-policy gradient

●●● · ~12 min

> REINFORCE ascends ∇J = G·∇log π(a|s): scale the log-probability gradient by the observed return. For a softmax policy over preferences h the gradient has a closed form: ∇_h log π(a) = e_a − π, a one-hot at the taken action minus the probability vector — pull up the taken action, push all actions down proportionally to their probability.

Implement `reinforce_grad(prefs, action, G)` → gradient vector of `G · log π(action)` with respect to the preference vector `prefs`, where π = softmax(prefs). Result: `G * (one_hot(action) − softmax(prefs))`.

<details><summary>Hint</summary>

Compute a stable softmax p of prefs, start from g = -p, add 1 at g[action], multiply by G.

</details>


In [ ]:
import numpy as np

def reinforce_grad(prefs, action, G):
    prefs = np.asarray(prefs, dtype=float)
    # your code
    return np.zeros(len(prefs))

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    k = int(rng.integers(2, 7))\n    prefs = rng.normal(0, 2, k)\n    if rng.random() < 0.5:\n        prefs[int(rng.integers(k))] = float(rng.choice([800.0, 950.0]))  # naive softmax -> NaN\n    return {\"prefs\": prefs, \"action\": int(rng.integers(k)),\n            \"G\": float(rng.normal(0, 3))}\n\ndef reinforce_grad(prefs, action, G):\n    h = np.asarray(prefs, dtype=float)\n    z = h - h.max()\n    p = np.exp(z); p = p / p.sum()\n    g = -p\n    g[action] += 1.0\n    return G * g", "reinforce_grad", 20)

---

**Go deeper**
- [Исходный урок (Microsoft, MIT)](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/6-Other/22-DeepRL)
- [Учебник ML (ШАД): обучение с подкреплением](https://contest.yandex.ru/tracks/ml/environment-interaction/reinforcement-learning)